<a href="https://colab.research.google.com/github/GreJangre/Baekjoon_Swift/blob/master/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations

import os
import sys
import json
import time
import argparse
from dataclasses import dataclass
from datetime import date, datetime, timedelta

import requests

BASE_URL = "http://data-dbg.krx.co.kr/svc/apis/sto"

DAILY_TRADE_ENDPOINTS = {
    "kospi": "/stk_bydd_trd",
    "kosdaq": "/ksq_bydd_trd",
    "konex": "/knx_bydd_trd"
}

MARKET_LABELS = {
    "kospi": "유가증권",
    "kosdaq": "코스닥",
    "konex": "코넥스"
}

OUTPUT_DIR = "krx_raw_json_30days"

@dataclass
class krxOpenApiClient:
  auth_key: str
  timeout: int = 30
  request_interval: float = 0.15

  def get_daily_trade_raw(self, market: str, bas_dd: str) -> dict:
    endpoint = DAILY_TRADE_ENDPOINTS[market]
    url = BASE_URL + endpoint
    headers = {"AUTH_KEY": self.auth_key.strip(), "Accept": "application/json"}
    resp = requests.get(url, headers=headers, params={"basDd": bas_dd}, timeout=self.timeout)
    time.sleep(self.request_interval)

    if resp.status_code != 200:
      raise RuntimeError(f"KRX API 오류 [{resp.status_code}] {endpoint} : {resp.text}")

    return resp.json()


def main():
  auth_key = os.environ.get("KRX_AUTH_KEY", "391F10D8791A47CFB31EF8F8BB010131DA0F3DAB")

  parser = argparse.ArgumentParser()
  parser.add_argument("--date",
                      dest="end_date",
                      default=None,
                      help="기준 종료일 YYYYMMDD (생략 시 오늘 날짜)"
                      )

  args, _unknown = parser.parse_known_args()

  if args.end_date:
    end_date = datetime.strptime(args.end_date, "%Y%m%d").date();
  else:
    end_date = date.today()

  start_date = end_date - timedelta(days=29)

  os.makedirs(OUTPUT_DIR, exist_ok=True)
  client = krxOpenApiClient(auth_key)

  print(f"조회 기간: {start_date.strftime('%Y-%m-%d')} ~ {end_date.strftime('%Y-%m-%d')} (요일 무관, 총 30일)")
  print(f"저장 폴더: {OUTPUT_DIR}/\n")

  summary = []

  d = start_date
  while d <= end_date:
    bas_dd = d.strftime("%Y%m%d")
    for market, label in MARKET_LABELS.items():
      try:
        raw = client.get_daily_trade_raw(market, bas_dd)
        row_count = len(raw.get("OutBlock_1", []))
        status = "OK"
      except Exception as e:
        raw = {"error": str(e)}
        row_count = 0
        status = "FAIL"

      out_path = os.path.join(OUTPUT_DIR, f"{market}_bydd_trd_{bas_dd}.json")
      with open(out_path, "w", encoding="utf-8") as f:
        json.dump(raw, f, ensure_ascii=False, indent=2)

      print(f"[{bas_dd}] {label:6s} -> {status}, {row_count}건, 저장: {out_path}")
      summary.append(
          {"date": bas_dd,
           "market": market,
           "status": status,
           "row_count": row_count
           }
      )

      d += timedelta(days=1)

if __name__ == "__main__":
  main()



조회 기간: 2026-07-29 ~ 2026-08-27 (요일 무관, 총 30일)
저장 폴더: krx_raw_json_30days/

[20260729] 유가증권   -> OK, 943건, 저장: krx_raw_json_30days/kospi_bydd_trd_20260729.json
[20260729] 코스닥    -> OK, 1821건, 저장: krx_raw_json_30days/kosdaq_bydd_trd_20260729.json
[20260729] 코넥스    -> OK, 109건, 저장: krx_raw_json_30days/konex_bydd_trd_20260729.json
[20260801] 유가증권   -> OK, 0건, 저장: krx_raw_json_30days/kospi_bydd_trd_20260801.json
[20260801] 코스닥    -> OK, 0건, 저장: krx_raw_json_30days/kosdaq_bydd_trd_20260801.json
[20260801] 코넥스    -> OK, 0건, 저장: krx_raw_json_30days/konex_bydd_trd_20260801.json
[20260804] 유가증권   -> OK, 943건, 저장: krx_raw_json_30days/kospi_bydd_trd_20260804.json
[20260804] 코스닥    -> OK, 1820건, 저장: krx_raw_json_30days/kosdaq_bydd_trd_20260804.json
[20260804] 코넥스    -> OK, 109건, 저장: krx_raw_json_30days/konex_bydd_trd_20260804.json
[20260807] 유가증권   -> OK, 943건, 저장: krx_raw_json_30days/kospi_bydd_trd_20260807.json
[20260807] 코스닥    -> OK, 1820건, 저장: krx_raw_json_30days/kosdaq_bydd_trd_20260807.json
